# 평가 노트북 (12_eval)

학습된 모델(또는 백본)을 벤치마크에 평가한다. CLI `scripts/eval.sh` 와 **동일한 로직**
(`project.evaluation.evaluate_on_benchmark_suite`)을 노트북에서 호출만 한다.

- 평가 설정: `BENCHMARK/configs/eval/*.yaml`
- 결과: `BENCHMARK/results/<recognizer.name>/`

> 핵심 로직은 `project/` 모듈에. 노트북은 *호출만* (SoT 일원화).

In [ ]:
# 셀 1) 환경 — GPU 지정(torch import 前 필수) + REPO 루트
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"   # ★ 빈 GPU 번호! nvidia-smi 로 확인. (GPU 0/1 풀이면 OOM)

import sys
from pathlib import Path
import yaml

REPO = Path.cwd()
while not (REPO / "project").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("REPO:", REPO, "| CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
# 셀 2) 평가 설정 — 여기만 바꿔서 쓴다
EVAL_CONFIG = REPO / "BENCHMARK/configs/eval/whisper_baseline.yaml"
BENCH_DATA = Path("/data/ASR/BENCHMARK/SILVER/GOLD")   # 벤치마크 데이터 루트 (<id>/transcript.jsonl)

# 학습 직후 outputs/<exp> 평가 시만 덮어쓰기 (아니면 None)
MODEL_PATH_OVERRIDE = None
NAME_OVERRIDE = None

cfg = yaml.safe_load(Path(EVAL_CONFIG).read_text())
rec = dict(cfg["recognizer"])
if MODEL_PATH_OVERRIDE:
    rec["model_path"] = str(MODEL_PATH_OVERRIDE)
if NAME_OVERRIDE:
    rec["name"] = NAME_OVERRIDE
print("recognizer:", rec["name"], "| type:", rec["type"], "| model_path:", rec["model_path"])
print("benchmarks:", cfg["benchmarks"], "| bench_data:", BENCH_DATA)

In [ ]:
# 셀 3) 어댑터 분기 → predict_fn (모델 로드)
if rec["type"] == "whisper":
    from project.data.adapters.whisper import build_predict_fn
    predict_fn = build_predict_fn(
        rec["model_path"],
        backbone=rec.get("backbone", rec["model_path"]),
        **rec.get("options", {}),
    )
elif rec["type"] == "sensevoice":
    from project.data.adapters.sensevoice import build_predict_fn
    predict_fn = build_predict_fn(rec["model_path"], **rec.get("options", {}))
else:
    raise ValueError(f"unknown recognizer.type: {rec['type']}")
print("predict_fn ready")

In [ ]:
# 셀 4) 벤치마크 ID → transcript.jsonl 경로 (Fail Fast)
root = Path(BENCH_DATA)
benchmark_paths = {}
for bid in cfg["benchmarks"]:
    p = root / bid / "transcript.jsonl"
    if not p.exists():
        raise FileNotFoundError(f"벤치마크 없음: {p}\n  - BENCH_DATA 경로 또는 벤치마크 ID 확인")
    benchmark_paths[bid] = str(p)
benchmark_paths

In [ ]:
# 셀 5) 평가 실행 → BENCHMARK/results/<name>__YYMMDD_HHMMSS/
import datetime
from project.evaluation import evaluate_on_benchmark_suite

ADD_TIMESTAMP = True   # True: 매 실행 따로 쌓임 / False: rec["name"] 그대로 (덮어쓰기)
run_name = rec["name"]
if ADD_TIMESTAMP:
    run_name = f"{rec['name']}__{datetime.datetime.now().strftime('%y%m%d_%H%M%S')}"

out_dir = REPO / "BENCHMARK" / "results" / run_name
results = evaluate_on_benchmark_suite(
    model_name=run_name,
    predict_fn=predict_fn,
    benchmark_paths=benchmark_paths,
    out_dir=out_dir,
    batch_size=cfg.get("batch_size", 16),
    save_diff=True,
)

print(f"{'benchmark':35s} {'CER%':>7} {'sCER%':>7} {'n':>6}")
print("-" * 60)
for bid, r in results.items():
    print(f"{bid:35s} {r.cer:7.2f} {r.scer:7.2f} {r.samples:6d}")
print("\n리포트:", out_dir)

In [ ]:
# 셀 6) (선택) 사람 읽는 리포트 + 오답 diff 미리보기
print((out_dir / "evaluation_report.txt").read_text())
print("\n--- diff (앞부분) ---")
diff = out_dir / f"{run_name}_diff.txt"
if diff.exists():
    print("\n".join(diff.read_text().splitlines()[:25]))

# ── 평가 후 GPU 메모리 해제 ──
import gc, torch
globals().pop('predict_fn', None)
gc.collect(); torch.cuda.empty_cache()
print('GPU 메모리 해제 (완전 해제는 Kernel Restart)')
